In [1]:
#Import libs
import pandas as pd
import numpy as np
from openai import OpenAI
import os
from anthropic import Anthropic
import json
from google import genai
from google.genai import errors
from openai import AsyncOpenAI
from anthropic import AsyncAnthropic
import asyncio
from itertools import product
import sys
from db_store_results import init_db, safe_parse_and_store, already_done, upsert_translation

In [2]:
from dotenv import load_dotenv

load_dotenv() 

True

Loading csv file with pilot candidates for translating and importing in SrpWN

In [3]:
#Load srpwn excel file
pilot_file = "./data/nedostajuci_sinsetovi.csv"

pilot_data = pd.read_csv(pilot_file)
pilot_data.shape

(200, 12)

In [4]:
#Load srpwn excel file
# check_file = "./data/SrpWN_translated_synsets.csv"

# check_file = pd.read_csv(check_file)
# check_file.shape

We are removing last 2 columns because they don't exist in SrpWN

In [5]:
candidates = pilot_data[pilot_data.columns[:-2]]

#### Prompt strategies: 
- Zero-shot
- Few-shot
- Chain-of-thought
- Prompt sa kontekstom
- Više kandidata

#### Models:
- GPT
- Claude
- Gemini


Define differnet model versions

In [6]:
models = {
    'openai': {'gpt-4o', 'gpt-5.5'},
    'anthropic': {'claude-opus-4-8', 'claude-sonnet-5'},
    'google': {'gemini-3.5-flash'}
}

strategies = ['zero-shot', 'few-shot', 'COT', 'with-context', 'multi-candidates']

combinations = [
    (version, model, strategy)
    for version, versions in models.items()
    for model, strategy in product(versions, strategies)
]

for c in combinations:
    print(c)

('openai', 'gpt-4o', 'zero-shot')
('openai', 'gpt-4o', 'few-shot')
('openai', 'gpt-4o', 'COT')
('openai', 'gpt-4o', 'with-context')
('openai', 'gpt-4o', 'multi-candidates')
('openai', 'gpt-5.5', 'zero-shot')
('openai', 'gpt-5.5', 'few-shot')
('openai', 'gpt-5.5', 'COT')
('openai', 'gpt-5.5', 'with-context')
('openai', 'gpt-5.5', 'multi-candidates')
('anthropic', 'claude-opus-4-8', 'zero-shot')
('anthropic', 'claude-opus-4-8', 'few-shot')
('anthropic', 'claude-opus-4-8', 'COT')
('anthropic', 'claude-opus-4-8', 'with-context')
('anthropic', 'claude-opus-4-8', 'multi-candidates')
('anthropic', 'claude-sonnet-5', 'zero-shot')
('anthropic', 'claude-sonnet-5', 'few-shot')
('anthropic', 'claude-sonnet-5', 'COT')
('anthropic', 'claude-sonnet-5', 'with-context')
('anthropic', 'claude-sonnet-5', 'multi-candidates')
('google', 'gemini-3.5-flash', 'zero-shot')
('google', 'gemini-3.5-flash', 'few-shot')
('google', 'gemini-3.5-flash', 'COT')
('google', 'gemini-3.5-flash', 'with-context')
('google', 

### API wrapper functions - GPT, Claude & Gemini
 In this section we define 3 helper fuctions to interact with different LLM providers:
 - `call_gpt(model, prompt)` - Sends request(prompt) to the OpenAI GPT API
 - `call_claude(model, prompt)` - Sends request(prompt) to the Anthropic Claude API
 - `call_gemini(prompt)` - Send request(prompt) to the Google Gemini API
 - model parameter defines different API version

All functions are designed to execute multiple tasks asynchronously

In [7]:
import asyncio
import os
from openai import AsyncOpenAI
from anthropic import AsyncAnthropic

openai_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
anthropic_client = AsyncAnthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
google_client = genai.Client(api_key = os.environ.get("GEMINI_API_KEY"))

#calling OpenAI model asynchronously
async def call_openai(model_name: str, prompt: str, strategy: str) -> dict:
     try:
        response = await openai_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.8
        )
        return {
            "model": model_name,
            "provider": "OpenAI",
            "strategy": strategy,
            "text": response.choices[0].message.content,
            "status": "success"
        }
     except Exception as e:
        return {"model": model_name, "provider": "OpenAI", "text": str(e), "strategy": strategy, "status": "error"}


#calling Anthropic model asynchronously
async def call_anthropic(model_name: str, prompt: str, strategy: str) -> dict:
    try:
        response = await anthropic_client.messages.create(
            model=model_name,
            max_tokens=1024*2,
            messages=[{"role": "user", "content": prompt}],
            # temperature=0.7
        )
        return {
            "model": model_name,
            "provider": "Anthropic",
            "strategy": strategy,
            # "text": response.content[0].text,
            "text": "".join(
                            block.text
                            for block in response.content
                            if block.type == "text"
                            ),
            "status": "success"
        }
    except Exception as e:
        return {"model": model_name, "provider": "Anthropic", "text": str(e), "strategy": strategy, "status": "error"}

#calling Google model asynchronously
async def call_google(model_name: str, prompt: str, strategy: str) -> dict:
    try:
        response = await google_client.aio.models.generate_content(
        model=model_name,
        contents=prompt
    )
        return {
            "model": model_name,
            "provider": "Google",
            "strategy": strategy,
            "text": response.text,
            "status": "success"
        }
    except Exception as e:
        return {"model": model_name, "provider": "Google", "text": str(e), "strategy": strategy, "status": "error"}


`get_prompt(row)` function receives 2 parameters - prompt strategy template and one record from WordNet. It formats the final prompt by substituting template pleaceholders with values extracted from the WordNet row.

In [8]:
def get_prompt(template, row):
    prompt = template.format(
        domain = row["domain"],
        literals = row["literals"],
        literal_id = row["ID"],
        definition = row['def'],
        usage = row["usage"],
        hypernymID = row["hypernymID"],
        hypernym_literals = row["hypernym_literals"],
        hypernym_def = row["hypernym_def"],
        hypernym_usage = row["hypernym_usage"] 
    )	

    return prompt

### PROMPT STRATEGIES
 
All prompts are stored in different template .txt files, located in ./prompt folder


#### 1. `load_template(strategy)` function receives strategy and opens appropriate txt template file

In [9]:
def load_template(strategy):
    with open(f"./prompts/{strategy}.txt", "r", encoding="utf-8") as f:
        template = f.read()
    return template

#### 2. Calling model API-s

In [10]:
async def call_apis(row):
    print(f"Translating synset {row.ID} ") 

    tasks = []
    for case in combinations:
         
        model = case[0]
        version = case[1]
        strategy = case[2]
        
        strategy_template = load_template(case[2])
        prompt = get_prompt(strategy_template, row)
        if already_done(conn, synset_id=row.ID, model=version, strategy=strategy):
            continue  # already done this combination, skip the API call 

        if case[0] == "openai":
            tasks.append(call_openai(case[1], prompt, case[2]))
        elif case[0] == "anthropic":
            tasks.append(call_anthropic(case[1], prompt, case[2]))
        elif case[0] == "google":
            tasks.append(call_google(case[1], prompt, case[2]))

        
    print(f"Sending prompt to {len(tasks)} models concurrently...")
    
    # Execute all API calls concurrently
    results = await asyncio.gather(*tasks)
    
    # Display comparison results
    print("\n" + "="*40 + "\nMODEL COMPARISON RESULTS\n" + "="*40)
    for res in results:
       
        print(f"\n[ {res['provider']} | {res['model']} ] Status: {res['status']}")
        print(f"Output:\n{res['text']}")
        print("-" * 30)
    return results

# asyncio.run(main())
# await call_apis(row)


#### 1.4. First results and thoughts

After evaluating different AI models using zero-shot prompt strategy based on the results from **JUST ONE** synset - ENG30-14327543-n - histamine headache; cluster headache - first conclusions are:

 - OpenAI models versions gpt-5 and gpt-5.5 gave similar and mostly valid translations. There are some punctuational differences and different choice of synonyms (rekurentna, ponvaljajuca).
 - OpenAI models gave wrong translation of a tehnical term (strucni naziv?) used in Serbian scientific literature (cefalalgija instead of cefalgija) ❌ 
  Anthropic Claude translated the term correctly ✅ (note: later in the testing is noticed that this model also sometimes return bad term) 
 - I'll give Claude model slight addvantage over Gpt for now, altough Gpt answer used complexed terrminology and sound more academic.


Next steps include testing models on bigger set and different prompt strategies.


#### 2.1. Testing Zero-shot strategy on 20 synsets, all models

In [11]:
sys.path.append(".")  # db_store_results.py treba da bude u istom folderu kao ovaj notebook

results_db_path = "./results/results.db"
conn = init_db(results_db_path)

In [16]:
## MULTIPLE SYNSETS
rows = pilot_data.iloc[0:20]
# rows = pilot_data[pilot_data.ID == 'ENG30-13551396-n']

for index, row in rows.iterrows():
    # print(row)
    result = await call_apis(row)

    for response_text in result:
        safe_parse_and_store(conn, synset_id=row["ID"], domain=row["domain"], api_result=response_text)

print("Saved to", results_db_path)

Translating synset ENG30-14327543-n 
Sending prompt to 0 models concurrently...

MODEL COMPARISON RESULTS
Translating synset ENG30-14289942-n 
Sending prompt to 0 models concurrently...

MODEL COMPARISON RESULTS
Translating synset ENG30-09782855-n 
Sending prompt to 0 models concurrently...

MODEL COMPARISON RESULTS
Translating synset ENG30-02380980-v 
Sending prompt to 0 models concurrently...

MODEL COMPARISON RESULTS
Translating synset ENG30-14159318-n 
Sending prompt to 0 models concurrently...

MODEL COMPARISON RESULTS
Translating synset ENG30-05724694-n 
Sending prompt to 2 models concurrently...

MODEL COMPARISON RESULTS

[ Google | gemini-3.5-flash ] Status: error
Output:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
------------------------------

[ Google | gemini-3.5-flash ] Status: error
Output:
503 UNAVAILABLE. {'error': {'cod

In [15]:
import sqlite3

conn = sqlite3.connect("./results/results.db")

df = pd.read_sql_query("SELECT * FROM translations WHERE status = 'success'", conn)
df.shape
df[df.synset_id == 'ENG30-14327543-n']

from db_store_results import progress_summary

dt = pd.read_sql_query("SELECT * FROM translations WHERE status = 'parse_error'", conn)
dt

df

# ENG30-13551396-n
# conn.execute("delete FROM translations WHERE strategy = 'COT' and synset_id = 'ENG30-14327543-n'")
# conn.commit()

,id,synset_id,domain,provider,model,strategy,status,literals,def_,usage,hypernymID,hypernym_literals,hypernym_def,hypernym_usage,raw_response,error_message,created_at
0,1,ENG30-14327543-n,medicine,OpenAI,gpt-4o,zero-shot,success,histaminska glavobolja; cluster glavobolja,bolna ponavljajuća glavobolja povezana sa oslo...,-,ENG30-14326607-n,glavobolja; cephalalgia,bol u glavi izazvan dilatacijom moždanih arter...,-,"```json\n{\n ""domain"": ""medicina"",\n ""BCS"": ...",None,2026-07-17T15:39:53.907310+00:00
1,2,ENG30-14327543-n,medicine,OpenAI,gpt-4o,few-shot,success,histaminska glavobolja; klaster glavobolja,"bolna, ponavljajuća glavobolja povezana sa osl...",-,ENG30-14326607-n,glavobolja; cefalička bol; cefalalgija,bol u glavi uzrokovan dilatacijom cerebralnih ...,-,"```json\n{\n ""domain"": ""medicine"",\n ""BCS"": ...",None,2026-07-17T15:39:53.909146+00:00
2,3,ENG30-14327543-n,medicine,OpenAI,gpt-5.5,zero-shot,success,histaminska glavobolja; klaster glavobolja,bolna rekurentna glavobolja povezana sa osloba...,-,ENG30-14326607-n,glavobolja; bol u glavi; cefalalgija,bol u glavi izazvan dilatacijom cerebralnih ar...,-,"{\n ""domain"": ""medicina"",\n ""BCS"": """",\n ""I...",None,2026-07-17T15:39:53.910060+00:00
3,4,ENG30-14327543-n,medicine,OpenAI,gpt-5.5,few-shot,success,histaminska glavobolja; klaster glavobolja; Ho...,bolna recidivirajuća glavobolja povezana sa os...,-,ENG30-14326607-n,glavobolja; cefalalgija,bol u glavi izazvan dilatacijom moždanih arter...,-,"{\n ""domain"": ""medicine"",\n ""BCS"": """",\n ""I...",None,2026-07-17T15:39:53.910945+00:00
4,5,ENG30-14327543-n,medicine,Anthropic,claude-opus-4-8,zero-shot,success,histaminska glavobolja; klaster glavobolja,bolna rekurentna glavobolja povezana sa osloba...,,ENG30-14326607-n,glavobolja; cefalalgija,bol u glavi izazvan dilatacijom cerebralnih ar...,,"{\n ""domain"" : ""medicina"",\n ""BCS"" : """",\n ...",None,2026-07-17T15:39:53.911895+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,989,ENG30-14327543-n,medicine,OpenAI,gpt-5.5,COT,success,histaminska glavobolja; klaster glavobolja,bolna ponavljajuća glavobolja povezana sa oslo...,-,ENG30-14326607-n,glavobolja; bol u glavi; cefalalgija,bol u glavi izazvan dilatacijom moždanih arter...,-,"{\n ""domain"": ""medicine"",\n ""BCS"": """",\n ""I...",None,2026-07-18T18:49:07.626527+00:00
437,990,ENG30-14327543-n,medicine,OpenAI,gpt-4o,COT,success,хистаминска главобоља; кластер главобоља,"болна, поновљена главобоља повезана са ослобађ...",-,,главобоља; цефалалгија,бол у глави изазван ширењем артерија мозга или...,-,"{\n ""domain"": ""medicine"",\n ""BCS"": """",\n ""I...",None,2026-07-18T18:49:07.630678+00:00
438,991,ENG30-14327543-n,medicine,Anthropic,claude-opus-4-8,COT,success,histaminska glavobolja; klaster glavobolja,"bolna, ponavljajuća glavobolja povezana sa osl...",-,ENG30-14326607-n,glavobolja; cefalalgija,bol u glavi uzrokovan proširenjem moždanih art...,-,"{\n ""domain"": ""medicine"",\n ""BCS"": """",\n ""I...",None,2026-07-18T18:49:07.632132+00:00
439,992,ENG30-14327543-n,medicine,Anthropic,claude-sonnet-5,COT,success,хистаминска главобоља; кластер главобоља,"болна главобоља која се понавља, повезана са о...",-,ENG30-14326607-n,главобоља; цефалалгија,бол у глави изазван проширењем можданих артери...,-,"{\n ""domain"": ""medicine"",\n ""BCS"": """",\n ""I...",None,2026-07-18T18:49:07.633249+00:00


In [17]:
# ukupno kandidata

# 5 modela x 5 strategija
# 25 * 20 primera = 500

from db_store_results import  export_to_excel
export_to_excel(conn, "./results/pilot_20_v1.xlsx", False)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
